# End-to-End Pipeline Basic 수치형·범주형 전처리, RandomOverSampler와 Logistic Regression을 하나의 Pipeline으로 구성하고, Pipeline 전체를 GridSearchCV에 전달했습니다. 선택 후 sealed Test를 한 번 평가하고 저장·reload 결과가 같은지 확인했습니다. 기본 실습 완료. 강의 문제 원문은 제외

In [2]:
from __future__ import annotations

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
n = 1200

# [1] 수치형 세 열과 범주형 한 열을 가진 원본 입력을 만듭니다.
# prompt_tokens에는 뒤에서 np.nan을 넣을 수 있도록 float dtype을 사용합니다.
X = pd.DataFrame({
    "prompt_tokens": rng.integers(20, 1800, n).astype(float),
    "retrieval_score": rng.normal(0.58, 0.18, n).clip(0, 1),
    "toxicity_score": rng.beta(1.5, 8.0, n),
    "route": rng.choice(
        ["chat", "rag", "agent"],
        n,
        p=[0.45, 0.40, 0.15],
    ),
})

# [2] 교육용 숨은 점수로 needs_review 정답을 만듭니다.
# 실제 서비스의 검토 정책이나 인과관계를 뜻하는 식은 아닙니다.
latent = (
    -5.0
    + 3.1 * (1 - X["retrieval_score"])
    + 5.0 * X["toxicity_score"]
    + 0.0010 * X["prompt_tokens"]
    + 0.8 * (X["route"] == "agent")
    + rng.normal(0, 0.9, n)
)
y = pd.Series(
    (latent > 0).astype(int),
    name="needs_review",
)

# [3] imputer가 실제로 작동하도록 일부 입력 셀을 비웁니다.
# 수치형 열마다 12개, route에서 8개의 결측값을 만듭니다.
for column in [
    "prompt_tokens",
    "retrieval_score",
    "toxicity_score",
]:
    missing_rows = rng.choice(n, size=12, replace=False)
    X.loc[missing_rows, column] = np.nan

X.loc[
    rng.choice(n, size=8, replace=False),
    "route",
] = np.nan

# [4] Pipeline 설정을 선택할 개발 데이터와 마지막 평가용 test를 분리합니다.
# test는 변수로 만들어 두지만 후보 선택 전에는 점수 계산에 사용하지 않습니다.
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("dev/test rows:", len(X_dev), len(X_test))
print("needs_review rate:", round(float(y_dev.mean()), 3))
print("missing values:", int(X.isna().sum().sum()))
print("sealed test used for scoring:", False)

assert X_dev.shape == (960, 4)
assert X_test.shape == (240, 4)
assert int(X.isna().sum().sum()) == 44

dev/test rows: 960 240
needs_review rate: 0.082
missing values: 44
sealed test used for scoring: False


In [3]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_review_pipeline():
    """혼합형 전처리·oversampling·분류기를 하나로 연결합니다."""
    numeric_columns = [
        "prompt_tokens",
        "retrieval_score",
        "toxicity_score",
    ]
    categorical_columns = ["route"]

    # 수치형 열은 중앙값으로 결측을 채운 뒤 단위를 표준화합니다.
    # 중앙값·평균·표준편차는 지금 계산하지 않고 fit 시점의 train에서 학습합니다.
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    # 범주형 열은 최빈값으로 결측을 채운 뒤 one-hot 열로 변환합니다.
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    # ColumnTransformer가 열 이름을 기준으로 서로 다른 경로를 적용합니다.
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ])

    # RandomOverSampler는 fit할 때만 소수 class의 기존 행을 복제합니다.
    # predict와 predict_proba에서는 sampler가 실행되지 않습니다.
    pipeline = ImbPipeline([
        ("preprocessor", preprocessor),
        ("sampler", RandomOverSampler(random_state=RANDOM_STATE)),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ])

    return pipeline


pipeline = build_review_pipeline()
print("pipeline steps:", [
    name for name, _ in pipeline.steps
])

assert [name for name, _ in pipeline.steps] == [
    "preprocessor",
    "sampler",
    "classifier",
]

pipeline steps: ['preprocessor', 'sampler', 'classifier']


In [4]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold


def fit_pipeline_search(pipeline, X_dev, y_dev):
    """개발 데이터 CV로 C를 선택하고 best Pipeline을 반환합니다."""
    cv = StratifiedKFold(
        n_splits=4,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    # Pipeline의 classifier 단계 안쪽 C를 세 값으로 비교합니다.
    search = GridSearchCV(
        estimator=pipeline,
        param_grid={
            "classifier__C": [0.1, 1.0, 10.0],
        },
        scoring="average_precision",
        cv=cv,
        n_jobs=1,
        refit=True,
    )

    # 후보 선택에는 개발 데이터만 사용합니다.
    search.fit(X_dev, y_dev)

    candidate_count = len(search.cv_results_["params"])
    cv_fit_count = candidate_count * cv.get_n_splits()

    assert candidate_count == 3
    assert cv_fit_count == 12

    return search, candidate_count, cv_fit_count


def evaluate_and_reload(
    search,
    X_test,
    y_test,
    new_requests,
):
    """test 1회 평가와 저장·reload 동등성 검사를 수행합니다."""
    # refit=True가 선택 후보를 X_dev 전체에 이미 다시 학습했습니다.
    best_pipeline = search.best_estimator_

    # 양성 class가 두 번째 열이라고 가정하지 않고 label 1의 위치를 찾습니다.
    classes = best_pipeline.named_steps[
        "classifier"
    ].classes_.tolist()
    positive_index = classes.index(1)

    # 후보 선택이 끝난 뒤 여기서 처음으로 sealed test 점수를 계산합니다.
    test_score = best_pipeline.predict_proba(
        X_test
    )[:, positive_index]
    test_ap = average_precision_score(y_test, test_score)

    # 새 요청은 정답이 없는 inference 입력입니다.
    before_score = best_pipeline.predict_proba(
        new_requests
    )[:, positive_index]
    before_prediction = best_pipeline.predict(new_requests)

    # 선택된 Pipeline 하나만 저장합니다.
    artifact_path = Path("review_pipeline.joblib")
    joblib.dump(best_pipeline, artifact_path)

    # joblib.load는 코드를 실행할 수 있으므로 신뢰된 파일만 불러옵니다.
    loaded_pipeline = joblib.load(artifact_path)
    loaded_classes = loaded_pipeline.named_steps[
        "classifier"
    ].classes_.tolist()
    loaded_positive_index = loaded_classes.index(1)

    after_score = loaded_pipeline.predict_proba(
        new_requests
    )[:, loaded_positive_index]
    after_prediction = loaded_pipeline.predict(new_requests)

    score_close = np.allclose(before_score, after_score)
    prediction_match = np.array_equal(
        before_prediction,
        after_prediction,
    )

    assert score_close
    assert prediction_match
    assert len(before_prediction) == len(new_requests)
    assert 0.0 <= test_ap <= 1.0

    return {
        "best_pipeline": best_pipeline,
        "test_ap": test_ap,
        "before_score": before_score,
        "before_prediction": before_prediction,
        "artifact_path": artifact_path,
        "score_close": score_close,
        "prediction_match": prediction_match,
    }


search, candidate_count, cv_fit_count = fit_pipeline_search(
    pipeline,
    X_dev,
    y_dev,
)

# 학습 때와 같은 네 열을 가진 새 요청 세 건입니다.
# 세 번째 prompt_tokens 결측은 Pipeline 안의 학습된 median으로 채워집니다.
new_requests = pd.DataFrame({
    "prompt_tokens": [180.0, 1250.0, np.nan],
    "retrieval_score": [0.91, 0.22, 0.48],
    "toxicity_score": [0.03, 0.51, 0.17],
    "route": ["chat", "agent", "rag"],
})

result = evaluate_and_reload(
    search,
    X_test,
    y_test,
    new_requests,
)

inference_table = pd.DataFrame({
    "needs_review_score": result["before_score"],
    "prediction": result["before_prediction"],
})

print("CV candidates/fits:", candidate_count, cv_fit_count)
print("best C:", search.best_params_["classifier__C"])
print("best CV AP:", f"{search.best_score_:.3f}")
print("sealed test AP:", f"{result['test_ap']:.3f}")
print(inference_table.to_string(
    index=False,
    formatters={
        "needs_review_score": "{:.3f}".format,
    },
))
print("artifact path:", result["artifact_path"])
print("reload scores close:", result["score_close"])
print(
    "reload predictions identical:",
    result["prediction_match"],
)

CV candidates/fits: 3 12
best C: 1.0
best CV AP: 0.440
sealed test AP: 0.631
needs_review_score  prediction
             0.001           0
             0.999           1
             0.286           0
artifact path: review_pipeline.joblib
reload scores close: True
reload predictions identical: True
